<a href="https://colab.research.google.com/github/Eliezer-Carvalho/Adamastor-GPT/blob/main/Adamastor_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Adamastor-GPT**

Este modelo é baseado na arquitetura <b> Transformer </b>, inspirado nos modelos do tipo <b> GPT (<i>Generative Pre-trained Transformer</i>)</b>. <br>

Ao longo deste Colab, iremos navegar pelas componentes cruciais dos modelos GPT-like, que atualmente dominam grande parte do mundo da Inteligência Artificial.

<b> No final, teremos um modelo estilo GPT funcional, capaz de realizar inferência de forma limpa e consistente. </b>

O treino vai ser realizado numa <b> T4 GPU</b>. <br>
Este Colab é extensão de um outro colab [Transformer - Attention Is All You Need](https://github.com/Eliezer-Carvalho/Adamastor-GPT/blob/main/transformer/Transformer%20-%20Attention%20Is%20All%20You%20Need.ipynb) que faz parte do rep [Adamastor-GPT](https://github.com/Eliezer-Carvalho/Adamastor-GPT). <br>
Todas as diferenças entre ambos os colabs estarão marcadas com uma ⭐.


In [99]:
!wget https://raw.githubusercontent.com/Eliezer-Carvalho/Adamastor-GPT/refs/heads/main/data/Os%20Lusiadas.txt

--2026-05-10 22:56:32--  https://raw.githubusercontent.com/Eliezer-Carvalho/Adamastor-GPT/refs/heads/main/data/Os%20Lusiadas.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 325114 (317K) [text/plain]
Saving to: ‘Os Lusiadas.txt.1’

Os Lusiadas.txt.1   100%[===================>] 317.49K  --.-KB/s    in 0.02s   

2026-05-10 22:56:32 (15.5 MB/s) - ‘Os Lusiadas.txt.1’ saved [325114/325114]



### **Libraries**

In [100]:
from tokenizers import Tokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = "cuda" if torch.cuda.is_available() else "cpu" #Modelos LLM rodam em GPU e não em CPU
print (device)

cuda


In [101]:
import re   #https://pypi.org/project/regex/

with open ("Os Lusiadas.txt", "r", encoding = "utf-8") as f:
  dataset = f.read()

#https://regexbox.com/cheatsheet
print (f"Caractéres --> {len(dataset)}")
print (f"Palavras --> {len(re.findall(r'\w+', dataset))}")
print (f"Linhas --> {len(re.findall(r'\n', dataset))}")

Caractéres --> 318104
Palavras --> 56947
Linhas --> 11044


### **Dados do Modelo**

In [102]:
Sequence_Length = 256
Batch_Size = 32
Embedding_Dimension = 64
Head_Size = 8 #Head_Size = Embedding_Dimension // Number_Head
Number_Head = 8
Blocos = 4

###


---

### **Tokenization**


In [103]:
tokenizer = Tokenizer.from_file ("/content/Os Lusiadas - Unigram - 22500")
dataset = tokenizer.encode(dataset)

tensor = torch.tensor (dataset.ids, dtype = torch.long) #Tensor Shape
print (tensor [:50]) #50 primeiros Tokens

n = int (0.9 * (len(tensor)))
dados_treino = tensor [:n] #90% Dados de Treino
dados_val = tensor [n:] #10% Dados de Teste

tensor([  618,  4077,     0,     0,    19,     0,  7105,   261,  3828,  1849,
           18,     0,  4462,  6474,   104,  1415,  2197,     0,    82,  2324,
         5649,  9114,    18,     0,  7157,    96,  1221, 17148,     8,  9972,
            1,     0,   103,  1892,    47,  4800, 16856,    18,     0,  1508,
        12273, 16594,     1,     0, 17627,   286,  3005, 16437,     0,  3306])


###


---

### **Batch e Sequence Length**

In [104]:
torch.manual_seed (2000)

def Batch (split):

  dados = dados_treino if split == 'train' else dados_val #Posteriormente no treino vai ser importante para testar o modelo com dados de treino e com dados de teste

  idx = torch.randint (0, len(dados) - Sequence_Length, (Batch_Size, ))
  x = torch.stack ([dados [i: i + Sequence_Length] for i in idx])
  y = torch.stack ([dados [i + 1: i + Sequence_Length + 1] for i in idx])

  x, y = x.to(device), y.to(device) #Passar os dados para a GPU

  return x, y

x, y = Batch('train')

print (f"(B, T) = {x.shape}") #(B, T)
print (x[1])

print (f"(B, T) = {y.shape}") #(B, T)
print (y[1])

(B, T) = torch.Size([32, 256])
tensor([   79,  1565, 20903,   737,     0,  5480, 17089,  8799,   706,  3012,
            4,     0,   593,  1815,  2348,    36,  1430, 13247,  4733,     1,
            0,   149,  5286, 14224,  7111,  9774,     1,     0,  6549, 14690,
        15989,  5286,  4341,     0, 14963,  6129,  3995,   198, 17048,  1604,
            0,     0,    11,    12,     0,   110, 17588, 15585,   270,     0,
           95,  6954,  3294,    48, 20338,    45, 10676,    26,     0,  1737,
          200,  9489,   891, 10680,     0,   218, 12526,  3120, 14584,     2,
            0, 10502,  1120,   139, 15345,  8457,     0,  1294,   525,    70,
         7816,   468,  2812,     0,  4618,  9877,  6242,  5658,     0,  2030,
        19582,   250,  3704,  1214,     2,     0,     0,    11,    13,     0,
        17199,  8260, 16294,    32,  3477,     1,     0,   348,  3715,  3653,
           45,   159,   287,    54,     0,    27, 18265, 12358, 15133,   200,
         4258,     1,     0,    2

###


---



### **Definição de uma Head**

Uma Head não é nada mais nada menos do que a aplicação do mecanismo <b> Attention</b>.

$$
{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

<p align = "center">
  <img src = "https://machinelearningmastery.com/wp-content/uploads/2021/09/tour_3.png" width = 250>
</p>



A <b> Head Size (tamanho da cabeça)</b> representa o profundidade que uma Single Head irá ter. <br>
Ou seja, cada head não “vê” a sequência inteira do embedding diretamente. <br>
Transforma os embeddings para um espaço mais pequeno (<b>head_size</b>), onde aprende relações diferentes. <br>

Desta maneira cada head aprende padrões diferentes (ex: sintático, semântico, posição, etc.)

$$
head{size} = \frac{d_{emb}}{heads}
$$

In [105]:
#Uso de POO, standard na indústria e mais prático
class Head (nn.Module): #Head = Self Masked Attention
  def __init__(self):
    super().__init__()
    self.Query = nn.Linear (Embedding_Dimension, Head_Size, bias = False) #Isto é praticamente uma MLP sem bias
    self.Key = nn.Linear (Embedding_Dimension, Head_Size, bias = False) #Isto é praticamente uma MLP sem bias
    self.Value = nn.Linear (Embedding_Dimension, Head_Size, bias = False) #Isto é praticamente uma MLP sem bias

    self.register_buffer('tril', torch.tril(torch.ones(Sequence_Length, Sequence_Length))) #Melhor assim, para não estar a criar a mask sempre, menos custo computacional

    self.Dropout = nn.Dropout (0.2) #Dropout é importante para evitar overfitting

  def forward (self, x):
    B, T, C = x.shape

    Q = self.Query (x)
    K = self.Key (x)
    V = self.Value (x)

    # Scaled Dot-Product Attention
    attention_score = Q @ K.transpose (-2, -1) * (Head_Size ** -0.5)

    attention_score = attention_score.masked_fill (self.tril [:T, :T] == 0, float ("-inf")) #Self Masked Attention

    attention_score = F.softmax (attention_score, dim = -1) #Normalização da attention_score

    attention_score = self.Dropout (attention_score)

    output = attention_score @ V
    return output

###


---


### **Definição de Multi Head** ⭐

É um dos conceitos mais simples e importantes dentro da arquitetura GPT.

Consiste em concatenar os outputs de várias <b> <i> heads (cabeças de atenção) </i> </b> e, em seguida, aplicar uma <i> <b> output projection </b> </i> para combinar e misturar a informação obtida.

As <b> Multi-Heads </b> permitem que o modelo capte diferentes tipos de relações dentro de uma sequência <b> (Sequence Length)</b>, analisando-a sob múltiplas perspetivas em simultâneo.

$$
{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O
$$

$$
{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)
$$

<p align = "center">
  <img src = "https://www.jeremyjordan.me/content/images/size/w1000/2023/05/multi-head-attention.png" width = "600">
</p>

In [106]:
class Multi_Head (nn.Module):
  def __init__(self):
    super().__init__()
    self.Concat = nn.ModuleList ([Head() for i in range (Number_Head)]) #Head Concatenation #Definição de Multi Head Attention #Concat o output do Número de Heads
    self.Output_Projection = nn.Linear (Embedding_Dimension, Embedding_Dimension) #Necessário para misturar a informação das Heads #Desta forma volta também à dimensão (B, T, C)
    self.Dropout = nn.Dropout (0.2)

  def forward (self, x):

    output = self.Output_Projection (torch.cat ([h (x) for h in self.Concat], dim = -1))
    output= self.Dropout(output)
    return output


###


---


### **Definição de um Block ([Pre-LayerNorm Transformer](https://medium.com/@ashutoshs81127/why-pre-norm-became-the-default-in-transformers-4229047e2620))** ⭐

Com o paper <i> “Attention Is All You Need”</i>, foi introduzida a arquitetura Transformer, originalmente com uma abordagem chamada <b> Post-Norm</b>. <br>
No entanto, rapidamente os investigadores identificaram problemas ao escalar modelos mais profundos, como:

- <b> Treino instável em redes profundas </b>
- <b> Gradientes a explodir ou a desaparecer </b>

Isto levou ao surgimento da abordagem <b> Pre-Norm</b>, que melhorou significativamente a estabilidade do treino.
<br>
<br>
#### <b> Post-Norm vs Pre-Norm ? </b>

- <b>Abordagem Post-Norm:</b>

<i> Attention -> Add -> Norm -> Feed Forward -> Add -> Norm</i>

- <b>Abordagem Pre-Norm:</b>

<i> Norm -> Attention -> Add -> Norm -> Feed Forward -> Add </i>

<br>

Nos modelos atuais, o bloco típico é <b> Pre-Norm</b>, e a ordem correta é:
<b>
- Layer Norm
- Multi-Head Self-Attention
- Residual Connection (Add)
- Layer Norm
- Feed Forward Neural Network (MLP)
- Residual Connection (Add)
</b>

Estes métodos conjugados é aquilo a que chamamos de <b> Block </b> ou <b> Layer</b>.
<br>

<p align = "center">
  <img src = "https://miro.medium.com/v2/resize:fit:1358/1*PGF3P2R9xaP7sB_NwmlNZw.png" width = 625>
</p>

In [107]:
class Block (nn.Module):
  def __init__(self):
    super().__init__()

    self.LayerNorm_Attention = nn.LayerNorm (Embedding_Dimension) #Layer Norm
    self.Masked_Multi_Head_Attention = Multi_Head() #Self Masked Attention


    self.LayerNorm_FNN = nn.LayerNorm (Embedding_Dimension) #Layer Norm
    self.FeedForward_NeuralNet = nn.Sequential ( #MLP
        nn.Linear (Embedding_Dimension, 4 * Embedding_Dimension),
        nn.GELU(),
        nn.Linear (Embedding_Dimension * 4, Embedding_Dimension),
        nn.Dropout (0.2)
    )


  def forward (self, x):
    Norm_Attention = self.Masked_Multi_Head_Attention (self.LayerNorm_Attention (x)) #Norm #LayerNorm
    Add_Attention = x + Norm_Attention #Add #Residual Connection
    x = Add_Attention

    Norm_FNN = self.FeedForward_NeuralNet (self.LayerNorm_FNN (x)) #Norm #LayerNorm
    Add_FNN = x + Norm_FNN #Add #Residual Connection
    x = Add_FNN

    return x

###


---


### **Modelo Adamastor-GPT** ⭐

<i> Implementação do Modelo Final </i>

In [108]:
class ADAMASTOR_GPT (nn.Module):
  def __init__(self):
    super().__init__()
    self.Embedding = nn.Embedding (tokenizer.get_vocab_size(), Embedding_Dimension) #Embedding dos Tokens
    self.Positional_Encoding = nn.Embedding (Sequence_Length, Embedding_Dimension) #Positional Encoding dos Tokens

    self.Blocks = nn.Sequential (*[Block() for i in range (Blocos)])

    self.LayerNorm = nn.LayerNorm(Embedding_Dimension) #Layer Norm antes do Language Modelling
    self.Language_Modeling = nn.Linear (Embedding_Dimension, tokenizer.get_vocab_size()) #Converte o vetor final do modelo em logits sobre o vocabulário inteiro.

  def forward (self, x):
    B, T = x.shape

    x = self.Embedding (x) + self.Positional_Encoding (torch.arange (T, device = x.device))

    x = self.Blocks(x)

    x = self.LayerNorm (x)

    logits = self.Language_Modeling(x)

    return logits #Return de um tensor (4, 8, 5000)

###


---


### **Treino do Modelo (Super-Importante)** ⭐

<b> Como já vimos no código, os pesos são inicializados de forma aleatória.</b> <br>
Assim, é essencial um <b> processo de treino</b>, que permita atualizar estes parâmetros iterativamente com base no erro do modelo, tornando-o progressivamente mais preciso.


Para treinar estes parâmetros, é necessário um processo de otimização baseado em <b>gradientes</b>. <br>
O modelo faz previsões e compara-as com os valores reais através de uma função de perda <i>(loss function)</i>. <br>
<br>

Uma das funções de perda mais utilizadas em problemas de linguagem é a <b> <i> Cross Entropy Loss</b></i>, que penaliza o modelo quando atribui baixa probabilidade à resposta correta. <br>
Em tarefas de linguagem, isto é essencial porque o objetivo é maximizar a probabilidade do próximo token correto. <br>
<br>

Depois de calcular a <i>loss</i>, utilizamos o processo de backpropagation para determinar os <b> Gradientes.</b> <br>
Os Gradientes representam a direção e a magnitude com que cada peso da rede deve ser ajustado para minimizar o erro. <br>
Em termos simples, indicam como cada parâmetro influencia o resultado final e como deve ser alterado para melhorar a performance do modelo.
<br>
<br>
Os gradientes por si só não atualizam os pesos! <br>
Essa tarefa é realizada pelos <b> Otimizadores</b>, que utilizam os gradientes para aplicar correções aos parâmetros de forma eficiente e estável. <br>
Em vez de simplesmente ajustar os pesos diretamente com base no erro, os otimizadores definem a melhor forma de os atualizar ao longo do tempo, tendo em conta fatores como a taxa de aprendizagem e o comportamento histórico dos gradientes. <br>
Entre os otimizadores mais utilizados estão o SGD (*Stochastic Gradient Descent*), o Adam e o AdamW.

<br>
<br>

<b> Resumo:</b> <br>
O treino de uma arquitetura Transformer segue este ciclo:
<b>
- Forward pass (previsão do modelo)
- Cálculo da Cross Entropy Loss
- Backpropagation (cálculo dos gradientes)
- Otimizador atualiza os pesos
- Repetição até o modelo convergir
</b>

#### *Função que Treina o modelo*

In [109]:
def training (modelo, steps):

  modelo.train() #Ativa o treino

  for i in range (steps):

    x, y = Batch('train') #(B, T)

    logits = modelo (x)

    #print (logits.shape) #(16,32,5000)
    B, T, V = logits.shape #(Batch, Sequence Length, Vocab_Size)

    loss = loss_function (
        logits.view (B * T, V),
        y.view (B * T)
    ) #Entrada que o Cross Entropy espera

    optimizer.zero_grad() #Zera os gradientes acumulados nos parâmetros.
    loss.backward() #Backpropagation

    optimizer.step() #Optimizer aplicado

    if i % 1000 == 0: #Printa de 1000 em 1000

      losses = loss_calc (modelo)
      print(f"Step {i} | Train Loss: {losses['train']:.4f} | Validation Loss: {losses['val']:.4f} | Perplexity Loss: {math.exp(loss.item()):.4f}")

#### *Calcular a Loss*

Esta função foi altamente influenciada por este rep -> [nanoGPT](#https://github.com/karpathy/nanoGPT/blob/master/train.py)

In [110]:
#https://github.com/karpathy/nanoGPT/blob/master/train.py
@torch.no_grad()
def loss_calc (modelo, loop_eval = 500):

    output = {}
    modelo.eval() #Desliga modo de treino

    for split in ['train', 'val']: #Para calcular a loss para os dois conjuntos de dados

        losses = torch.zeros (loop_eval) #Cria um tensor com int "0" com tamanho loop_eval

        for k in range (loop_eval): #Por cada loop_eval o modelo vai ser alimentado por um novo batch de dados e vai ser testado a sua habilidade tanto em dados de teste como em dados de validação

            x, y = Batch (split) #Quando for train:                         #Quando for val:
            logits = modelo (x) #Alimenta o modelo com dados de treino      #Alimenta o modelo com dados de teste

            B, T, V = logits.shape

            loss = loss_function (
                logits.view (B * T, V),
                y.view (B * T)
            )

            losses[k] = loss.item() #Cria uma lista com as losses todas

        output[split] = losses.mean() #Calcula a média, mais preciso

    modelo.train() #Volto ao modo de treino
    return output

### **Adamastor-GPT Treino**

Só para ter noção, o modelo base apresentado pelo paper *Attention Is All You Need* foi treinado com 100k steps. :)

In [111]:
modelo = ADAMASTOR_GPT().to(device)
#print (modelo)
optimizer = torch.optim.AdamW (modelo.parameters(), lr = 3e-4) #Usa-se AdamW ou SGD #Responsável por atualizar os pesos do modelo
#print (optimizer)
loss_function = nn.CrossEntropyLoss() #Função de perda que mede quão diferente está a previsão do modelo da resposta correta.


training (modelo, 5000)

Step 0 | Train Loss: 10.1126 | Validation Loss: 10.1204 | Perplexity Loss: 25562.0175
Step 1000 | Train Loss: 5.9534 | Validation Loss: 8.4912 | Perplexity Loss: 383.7966
Step 2000 | Train Loss: 3.5581 | Validation Loss: 9.6322 | Perplexity Loss: 37.8877
Step 3000 | Train Loss: 2.5931 | Validation Loss: 11.0750 | Perplexity Loss: 14.9241
Step 4000 | Train Loss: 2.1507 | Validation Loss: 12.1079 | Perplexity Loss: 9.4724


###


---

### **Geração de Texto** ⭐

Esta função foi altamente influenciada por este rep -> [nanoGPT](#https://github.com/karpathy/nanoGPT/blob/master/train.py)

In [112]:
@torch.no_grad()
def gerar_texto (modelo, max_tokens, indice, temperatura = 1, top_tokens = None):

  modelo.eval() #Desliga o treino

  for i in range (max_tokens): #Vai adicionando token a token

    indice_condition = indice [:, -Sequence_Length:] #Alimento o modelo com o máximo de tokens possíveis, não posso dar um contexto maior do que daquele com que treinou

    logits = modelo (indice_condition) #Aplicação da Arquitetura

    logits = logits [:, -1, :] / temperatura #Foca no que gerou anteriormente #Temperatura aumenta ou diminui a creatividade. Se > 1 há mais chance de tokens raros serem escolhidos

    #https://github.com/karpathy/nanoGPT/blob/master/model.py
    if top_tokens is not None: #TopK funciona como um limitador que ajuda o modelo a olhar para os tokens com  melhores probabilidades, em vez de olhar para todos
              V = torch.topk (logits, min(top_tokens, logits.size (-1))) #https://docs.pytorch.org/docs/2.11/generated/torch.topk.html
              logits [logits < V [:, [-1]]] = -float('Inf') #Substitui os tokens com menor probabilidade por -inf


    probs = F.softmax (logits, dim = -1) #Aplica Softmax para ter as probabilidades

    prev = torch.multinomial (probs, num_samples = 1) #Seleciona um token

    indice = torch.cat ((indice, prev), dim  = 1) #Adiciona esse token ao contexto

  return indice

### **Adamastor-GPT Inferência**

In [113]:
contexto = torch.zeros ((1, 1), dtype = torch.long, device = device) #Cria um tensor [1,1] com o valor 0 e à medida que entra na função vai adicionando tokens
#print (contexto)
indice = gerar_texto (modelo, max_tokens = Sequence_Length, indice = contexto)

#print (indice)
#print (indice.shape)

#texto = Tokenizer.decode(indice[0].tolist())
#texto = texto.replace(" ", "  ")
#print(texto)
#print (decode(encode((indice [0,1:].tolist()))))

print (tokenizer.decode(indice[0].tolist()))


 3 
 "Assim com  firme  em quanto  maquinava o Bétis  de sangue  desparzido ,  pelejando 
 Ou n os amostra a  terra que busca n a lira  nome e fama 
 E com razões  me louva  est oc adas, 
 A terra  "Sabe que  há  que lhe ensinas se i, minha s amigas , 
 E já  a mãe de  Men ino  as alma s acendia . 
 8 4 
 "Não  consentiu  a morte  indina 
 "Mais  leg on  a luz  trazendo  sempre  sopra  Eolo , 
 Vencidos  e em  miséria  extrema  postos  n em defesa 
 A  vender  pelo  rosto angélico  apertava m; 
 D o Olimpo  estrelas  como  in ópia, , 
 Pel os sinais  levando  do  que achar ias , ou  Bóreas  injuriado, 
 Outro  de  Atenas , I os, A rgo  e Sala mina ndo. 
 Com que a  torne a  tomar 
 Onde  o Cabo A rsin ário  o nome  e a terra  espanta , 
 A boca  negra,  os dentes  amarelo . 
 
 4 0 
 Do mal  que  nomeada . 
 E não se  acabará , pe noso e    todos assentados 
 Fazer  u se  senti  de tal  arte, 
 Ele,  ador ação, 
 Co'o sangue  próprio , que se  estima  e   adormece s! 
 Se vê,  e de  s

#### Número de Parâmetros do Modelo

In [114]:
num_params = sum(p.numel() for p in modelo.parameters())
tam_na_gpu = num_params * 4

tam_na_gpu_train = tam_na_gpu * 4

print (num_params)
print (f"{tam_na_gpu:.0f} Bytes")
print (f"{tam_na_gpu_train:.0f} Bytes")
#save = torch.save(modelo.state_dict(), "Adamastor-GPT.pth")

3001564
12006256 Bytes
48025024 Bytes


### Características do Modelo

In [115]:
'''
print (modelo.state_dict())
for param in modelo.state_dict():
  print (param)
'''

'\nprint (modelo.state_dict())\nfor param in modelo.state_dict():\n  print (param)\n'

###


---

## Notas

O teu ADAMASTOR_GPT é:

um Transformer decoder-only (tipo GPT)
com:
4 camadas (blocos)
8 cabeças de atenção
embeddings de tamanho 64
vocabulário de 5000 tokens
contexto máximo de 32 tokens

FAZER ESTA LEITURA

interpretar loss value, o que o valor quer dizer ? <br>

CPU do PC vs TPU - tempo a treinar o modelo e resultados obtidos

Dropout é uma técnica de regularização usada para evitar overfitting.